See [Lifecycle hooks](../../code/lifecycle_hooks.md) for all hook names. `MirrorDirectoryPlugin` (example code in the next cell — not part of the `labmate` package) registers on `aqm.hooks` via `.attach(...)`.

In [ ]:
from __future__ import annotations

import logging
import shutil
from concurrent.futures import Future, ThreadPoolExecutor
from pathlib import Path
from typing import Any, Union

from labmate.acquisition import LifecycleHooks, NotebookAcquisitionData


logger = logging.getLogger(__name__)


class MirrorDirectoryPlugin:
    """Mirrors acquisition files to a second directory and can restore ``.h5`` before analysis."""

    def __init__(self, mirror_root: Union[str, Path]) -> None:
        self._mirror_root = Path(mirror_root)

    def attach(self, hooks: LifecycleHooks) -> None:
        hooks.add_acquisition_saved(self._mirror_after_persist)
        hooks.add_figure_saved(self._mirror_after_persist)
        hooks.add_analysis_data_loading(self._on_analysis_data_loading)

    def _mirror_after_persist(self, acquisition: NotebookAcquisitionData, **kwargs: Any) -> None:
        """Schedule mirror copy on a worker thread so the notebook thread is not blocked."""

        executor = ThreadPoolExecutor(max_workers=1)

        def copy_job() -> None:
            self._mirror_copy_to_destination(acquisition)

        def shutdown_executor(_future: Future) -> None:
            executor.shutdown(wait=False)

        future = executor.submit(copy_job)
        future.add_done_callback(shutdown_executor)

    def _mirror_copy_to_destination(self, acquisition: NotebookAcquisitionData) -> None:
        source_prefix = Path(acquisition.filepath)
        source_dir = source_prefix.parent
        prefix_name = source_prefix.name

        destination_dir = self._mirror_root / source_dir.name
        destination_dir.mkdir(parents=True, exist_ok=True)

        for src in source_dir.glob(f"{prefix_name}*"):
            if src.is_file():
                shutil.copy2(src, destination_dir / src.name)

    def _on_analysis_data_loading(self, full_h5_path: str, **kwargs: Any) -> None:
        """Restore missing local ``.h5`` from mirror; must stay synchronous before open."""

        target = Path(full_h5_path)
        if target.is_file():
            return

        mirror_path = self._mirror_root / target.parent.name / target.name

        if not mirror_path.is_file():
            return

        logger.info("Restoring %s from mirror at %s", target, self._mirror_root)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(mirror_path, target)

In [ ]:
import numpy as np

from labmate.acquisition_notebook import AcquisitionAnalysisManager

In [ ]:
data_dir = Path().resolve()
mirror_dir = Path().resolve()

aqm = AcquisitionAnalysisManager(str(data_dir))
MirrorDirectoryPlugin(mirror_dir).attach(aqm.hooks)

In [ ]:
aqm.acquisition_cell("Experience 02")

x = np.linspace(0, 10, 100)
y = np.sin(x)

aqm.save_acquisition(x=x, y=y)

In [ ]:
# Retrieve data from data_dir
aqm.analysis_cell()

Suppress .h5 file in data_dir, and keep it in mirror_dir, then try to run

In [ ]:
# Retrieve data from mirror_dir if data from data_dir are not available or suppressed.
# analysis_cell runs analysis_data_loading hooks first; MirrorDirectoryPlugin can restore the .h5.
aqm.analysis_cell("path to your .h5 file in data_dir")